# 08 — Optuna Hyperparameter Search

**Goal:** Replace the manual 4-config grid with an Optuna random search. The objective function uses **walk-forward CV** (5 expanding folds from Step 1) and maximizes the **mean OOF Sharpe Ratio**.

**Search Space:**
- `lr`: log-uniform [1e-4, 1e-2]
- `units`: categorical [32, 64, 128]
- `dropout`: categorical [0.0, 0.2, 0.4]
- `num_layers`: categorical [1, 2]

**Pruning:** `MedianPruner` kills underperforming trials after each fold, saving compute.

In [ ]:
import sys, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Load features from Notebook 03
with (INTERIM / 'features_for_lstm.pkl').open('rb') as f:
    bundle = pickle.load(f)

X = np.concatenate([bundle['train_x'], bundle['val_x'], bundle['test_x']], axis=0)
y = np.concatenate([bundle['train_y'], bundle['val_y'], bundle['test_y']], axis=0)
dates = np.concatenate([bundle['train_dates'], bundle['val_dates'], bundle['test_dates']], axis=0)

# Reconstruct close across the full window
merged = pd.read_parquet(INTERIM / 'merged_with_llm_sentiment.parquet').sort_values('date').reset_index(drop=True)
close = merged['close'].values

n_features = X.shape[-1]
print(f'X: {X.shape}  y: {y.shape}  close: {close.shape}  n_features: {n_features}')

# Class weights (computed on full set)
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y)
weights = compute_class_weight('balanced', classes=classes, y=y)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print(f'Class weights: {class_weight}')

## 8.1 Run Optuna search
Each trial trains 5 LSTM models (one per walk-forward fold) with EarlyStopping. The `MedianPruner` kills trials whose cumulative OOF Sharpe is below the median after fold 2.

**Note:** On CPU, 15 trials × 5 folds × 15 epochs takes ~10-15 min. On Colab T4 GPU, ~3-5 min.

In [ ]:
from src.cv.optuna_search import run_optuna_search

best_params = run_optuna_search(
    X=X, y=y, close=close, dates=dates,
    n_features=n_features,
    n_trials=15,
    n_folds=5,
    min_train=400,
    val_size=60,
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    fee=0.001,
    threshold=0.5,
    pruner_n_startup=3,
    pruner_n_warmup=2,
    output_path=OUTPUTS / 'best_optuna_params.json',
    seed=42,
)

## 8.2 Inspect the best params

In [ ]:
with (OUTPUTS / 'best_optuna_params.json').open() as f:
    best = json.load(f)
print(json.dumps(best, indent=2))

## 8.3 Summary
- Optuna search over `lr`, `units`, `dropout`, `num_layers` with walk-forward CV objective.
- MedianPruner killed underperforming trials after fold 2.
- Best params saved to `outputs/best_optuna_params.json`.
- These params will be used by Notebook 09 (risk-managed backtest) and Notebook 10 (SHAP).